# FRESHOR_NOT Training on Google Colab

This notebook runs stage-1 detector training for the FRESHOR_NOT repository in Colab and saves model artifacts to Google Drive.

## 1) Enable Google Colab Runtime and GPU/TPU

Before running, set Runtime -> Change runtime type -> Hardware accelerator -> GPU.

Then run the next cell to verify the environment.

In [ ]:
import os
import sys
import platform
import subprocess
from pathlib import Path

print('Python:', sys.version)
print('Platform:', platform.platform())

# GPU check
result = subprocess.run(['bash', '-lc', 'nvidia-smi || true'], capture_output=True, text=True)
print(result.stdout if result.stdout.strip() else 'No NVIDIA GPU visible. If expected, switch Colab runtime to GPU.')

## 2) Clone the Repository in Colab

Set your repository details, then clone and switch to the desired branch.

In [ ]:
# Edit these values if needed
REPO_URL = 'https://github.com/amohan84/FRESHOR_NOT.git'
REPO_BRANCH = 'main'
WORKDIR = Path('/content/FRESHOR_NOT')

if WORKDIR.exists():
    subprocess.run(['bash', '-lc', f'rm -rf {WORKDIR}'], check=False)

subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
os.chdir(WORKDIR)
subprocess.run(['git', 'checkout', REPO_BRANCH], check=True)

print('Working directory:', os.getcwd())
subprocess.run(['bash', '-lc', 'ls -la | head -n 40'], check=False)

## 3) Install Python Dependencies

In [ ]:
subprocess.run(['python', '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-r', 'requirements.txt'], check=True)
# Fallback explicit install for Ultralytics if required
subprocess.run(['python', '-m', 'pip', 'install', 'ultralytics>=8.2.0'], check=True)

import ultralytics
print('Ultralytics version:', ultralytics.__version__)

## 4) Configure Environment Variables and Paths

Adjust Google Drive paths for your data and outputs. The dataset root should contain data_v2.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Update these paths for your Drive layout
DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/FRESHOR_NOT_COLAB')
DRIVE_DATA_ROOT = DRIVE_PROJECT_ROOT / 'data_v2'
DRIVE_OUTPUT_ROOT = DRIVE_PROJECT_ROOT / 'outputs'
DRIVE_CHECKPOINT = DRIVE_PROJECT_ROOT / 'checkpoints' / 'last.pt'  # optional

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
(DRIVE_PROJECT_ROOT / 'checkpoints').mkdir(parents=True, exist_ok=True)

# Ensure local project imports work
sys.path.append(str(WORKDIR))
os.environ['PYTHONPATH'] = f"{WORKDIR}:{os.environ.get('PYTHONPATH', '')}"

print('Drive project root:', DRIVE_PROJECT_ROOT)
print('Drive data root exists:', DRIVE_DATA_ROOT.exists())

In [ ]:
# Link data_v2 from Drive into the repo (preferred to avoid big copies)
local_data_v2 = WORKDIR / 'data_v2'
if local_data_v2.exists() or local_data_v2.is_symlink():
    subprocess.run(['bash', '-lc', f'rm -rf {local_data_v2}'], check=False)

if not DRIVE_DATA_ROOT.exists():
    raise FileNotFoundError(f'Expected dataset at {DRIVE_DATA_ROOT}. Upload/sync data_v2 there first.')

subprocess.run(['ln', '-s', str(DRIVE_DATA_ROOT), str(local_data_v2)], check=True)
print('Linked data_v2 ->', local_data_v2)
subprocess.run(['bash', '-lc', f'ls -la {local_data_v2} | head -n 20'], check=False)

## 5) Run Core Notebook/Script Cells

Configure training hyperparameters and run either fresh training or resume from a checkpoint.

In [ ]:
# Training config
EPOCHS = 15
IMGSZ = 640
BATCH = 16
DEVICE = '0'  # GPU in Colab
WORKERS = 2
TRAIN_FRACTION = 1.0
SEED = 42

train_cmd = [
    'python', 'train/train_stage1_detector_yolo.py',
    '--epochs', str(EPOCHS),
    '--imgsz', str(IMGSZ),
    '--batch', str(BATCH),
    '--device', DEVICE,
    '--workers', str(WORKERS),
    '--train-fraction', str(TRAIN_FRACTION),
    '--seed', str(SEED),
]
print('Command:', ' '.join(train_cmd))

In [ ]:
# Option A: fresh run via project training script
subprocess.run(train_cmd, check=True)

In [ ]:
# Option B: resume YOLO directly from checkpoint (use instead of Option A when resuming)
# If DRIVE_CHECKPOINT exists, copy it into the expected run path and resume.
resume_src = DRIVE_CHECKPOINT
resume_dst = WORKDIR / 'runs' / 'detect' / 'train3' / 'weights' / 'last.pt'
resume_dst.parent.mkdir(parents=True, exist_ok=True)

if resume_src.exists():
    subprocess.run(['cp', str(resume_src), str(resume_dst)], check=True)
    from ultralytics import YOLO
    model = YOLO(str(resume_dst))
    model.train(resume=True)
    print('Resumed from:', resume_dst)
else:
    print('No checkpoint found at', resume_src, '- skip this cell unless resuming.')

## 6) Validate Outputs in Colab

In [ ]:
# Validate key outputs exist
best_candidates = list((WORKDIR / 'runs' / 'detect').glob('**/weights/best.pt'))
last_candidates = list((WORKDIR / 'runs' / 'detect').glob('**/weights/last.pt'))
results_csv = list((WORKDIR / 'runs' / 'detect').glob('**/results.csv'))

assert len(last_candidates) > 0, 'No last.pt found in runs/detect'
assert len(best_candidates) > 0, 'No best.pt found in runs/detect'
assert len(results_csv) > 0, 'No results.csv found in runs/detect'

print('Found best.pt files:', len(best_candidates))
print('Found last.pt files:', len(last_candidates))
print('Found results.csv files:', len(results_csv))
print('Most recent best.pt:', sorted(best_candidates, key=lambda p: p.stat().st_mtime)[-1])

## 7) Save Artifacts to Google Drive

Copy trained weights, logs, and charts to Drive for persistent storage.

In [ ]:
import shutil

detect_dir = WORKDIR / 'runs' / 'detect'
latest_results = sorted(detect_dir.glob('**/results.csv'), key=lambda p: p.stat().st_mtime)
if not latest_results:
    raise FileNotFoundError('No results.csv found under runs/detect')

latest_run_dir = latest_results[-1].parent
artifact_dir = DRIVE_OUTPUT_ROOT / latest_run_dir.name
artifact_dir.mkdir(parents=True, exist_ok=True)

# Copy entire run folder
shutil.copytree(latest_run_dir, artifact_dir, dirs_exist_ok=True)

# Also copy best/last to a stable checkpoints location
best_pt = latest_run_dir / 'weights' / 'best.pt'
last_pt = latest_run_dir / 'weights' / 'last.pt'
if best_pt.exists():
    shutil.copy2(best_pt, DRIVE_PROJECT_ROOT / 'checkpoints' / 'best.pt')
if last_pt.exists():
    shutil.copy2(last_pt, DRIVE_PROJECT_ROOT / 'checkpoints' / 'last.pt')

print('Saved run to:', artifact_dir)
print('Checkpoint best:', (DRIVE_PROJECT_ROOT / 'checkpoints' / 'best.pt').exists())
print('Checkpoint last:', (DRIVE_PROJECT_ROOT / 'checkpoints' / 'last.pt').exists())

## 8) Export and Share the Notebook

Create a zip archive of outputs for easy download/share from Colab.

In [ ]:
archive_path = DRIVE_OUTPUT_ROOT / 'latest_stage1_artifacts.zip'
subprocess.run(['bash', '-lc', f'cd {DRIVE_OUTPUT_ROOT} && zip -r {archive_path.name} .'], check=False)
print('Archive path:', archive_path)
print('If zip utility is missing, use the copied folder directly in Drive.')

In [ ]:
# Quick view of Drive output tree
subprocess.run(['bash', '-lc', f'find {DRIVE_PROJECT_ROOT} -maxdepth 3 -type f | head -n 80'], check=False)